# Verificacion de imagen captura con la camara

In [1]:
# Esto lo tengo yo porque tengo problemas con la librería MKL y me da error de threading (Se me muere el kernel de jupyter)

import os
os.environ["MKL_THREADING_LAYER"] = "SEQUENTIAL"

%matplotlib inline

In [2]:
import cv2
import numpy as np


cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Extraccion de canales RGB
    b_channel = frame.copy()
    b_channel[:, :, 1] = 0  # Set G channel to 0
    b_channel[:, :, 2] = 0  # Set R channel to 0

    g_channel = frame.copy()
    g_channel[:, :, 0] = 0  # Set B channel to 0
    g_channel[:, :, 2] = 0  # Set R channel to 0
    
    r_channel = frame.copy()
    r_channel[:, :, 0] = 0  # Set B channel to 0
    r_channel[:, :, 1] = 0  # Set G channel to 0

    # Concatenar los canales y el original vertical y horizontal
    row_1_concat = cv2.hconcat([frame, b_channel])
    row_2_concat = cv2.hconcat([g_channel, r_channel])
    final_concat = cv2.vconcat([row_1_concat, row_2_concat])
    
    # Mostrar la imagen concatenada
    cv2.imshow('Original', final_concat)

    # Presiona la tecla 'q' o 'esc' para salir del bucle ¡En minuscula!
    key = cv2.waitKey(1) & 0xFF

    if key == ord('q') or key == 27:
        break   

# Release the video capture and close all windows
cap.release()
cv2.destroyAllWindows()

# Verificacion de que el sistema de aprendizaje profundo pueda detectar un rostro

Inicializacion de los modelos a utilizar

In [3]:
from facenet_pytorch import MTCNN, InceptionResnetV1
import torch

# Initialize MTCNN for face detection
mtcnn = MTCNN(image_size=160, margin=0, post_process=False)

# Initialize InceptionResnetV1 for recognition
resnet = InceptionResnetV1(pretrained='vggface2').eval()


c:\Users\Lucho_TUF_A14\anaconda3\envs\UNIE_CNN\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

# Carga una imagen de ejemplo para probar el recorte de cara, cargala tu. No preguntes como jeje.
img = cv2.imread(<image_path>)  # Reemplaza <image_path> con la ruta de tu imagen

# Test de face cropping
img_cropped = mtcnn(img, save_path='result/cropped.png')
img2show = img_cropped.permute(1, 2, 0).int().numpy()  
img2show = img2show[:, :, ::-1]  

plt.imshow(img2show)

def cropFace(img):
    img_cropped = mtcnn(img, save_path='result/cropped.png')
    if img_cropped is None:
        return None
    img2show = img_cropped.permute(1, 2, 0).int().numpy()  
    img2show = img2show[:, :, ::-1]  
    return img2show.astype(np.uint8)  

AttributeError: 'NoneType' object has no attribute 'size'

In [ ]:


cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    cropped_face = cropFace(frame)
    size = min(frame.shape[0], frame.shape[1])
    if cropped_face is not None:
        cropped_face_resized = cv2.resize(cropped_face, (size, size))  # Resize to 160x160 for the model

    frame_concatenated = cv2.hconcat([frame, cropped_face_resized])
    cv2.imshow('Pitufo CAM', frame_concatenated)

    # Presiona la tecla 'q' o 'esc' para salir del bucle ¡En minuscula!
    key = cv2.waitKey(1) & 0xFF

    if key == ord('q') or key == 27:
        break

cap.release()
cv2.destroyAllWindows()